# 03 — Extraction & Evaluation

This notebook visualizes the **OCR output** and **structured extraction**
from the MedVault pipeline, and demonstrates the evaluation metrics (CER, WER)
implemented in `evaluation/metrics.py`.

We will cover:
1. Running the full pipeline on a sample image
2. Inspecting the raw OCR output
3. Viewing the structured extraction (lab_results JSON)
4. Computing CER and WER against ground truth
5. Running the benchmark script

## Setup & Imports

In [ ]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'backend'))

import matplotlib.pyplot as plt

from backend.pipeline import run_pipeline
from evaluation.metrics import character_error_rate, word_error_rate, exact_match_accuracy

%matplotlib inline
print('Imports successful')

## 1. Run the Full Pipeline

We'll run the unified `run_pipeline` function on a sample image. This
executes the full DAG: preprocess -> classify -> OCR -> extract -> validate.

If no sample image is available, we'll create a synthetic one.

In [ ]:
import cv2
import numpy as np

IMAGE_PATH = str(ROOT / 'tests' / 'sample_images' / 'sample.png')

if not Path(IMAGE_PATH).exists():
    print('Sample not found; creating synthetic image...')
    synthetic = np.ones((800, 600, 3), dtype=np.uint8) * 240
    cv2.putText(synthetic, 'LAB REPORT', (150, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 2)
    cv2.putText(synthetic, 'ALT  45  U/L', (150, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)
    cv2.putText(synthetic, 'AST  32  U/L', (150, 260), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)
    cv2.putText(synthetic, 'BILIRUBIN  0.8  mg/dL', (150, 320), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)
    IMAGE_PATH = str(ROOT / 'notebooks' / '_sample_synthetic.png')
    cv2.imwrite(IMAGE_PATH, synthetic)

print(f'Running pipeline on: {IMAGE_PATH}')
result = run_pipeline(IMAGE_PATH, include_summary=False, include_evaluation=False)
result_dict = result.to_dict()
print('Pipeline completed successfully!')

## 2. Inspect the Raw OCR Output

The pipeline result contains the raw OCR text, the engine used, and
confidence scores. Let's inspect these.

In [ ]:
# Show the top-level keys
print('Top-level result keys:')
for key in result_dict:
    print(f'  - {key}')

# Show OCR-specific info if available
if 'ocr' in result_dict:
    ocr = result_dict['ocr']
    print(f'\n=== OCR Stage ===')
    print(f'Engine: {ocr.get("ocr_engine_used", "N/A")}')
    print(f'Doc class: {ocr.get("doc_class", "N/A")}')
    print(f'Confidence: {ocr.get("confidence", "N/A")}')
    print(f'Processing time: {ocr.get("processing_time_seconds", "N/A")}s')
    raw = ocr.get('raw_output', '')
    print(f'\nRaw OCR output (first 500 chars):')
    print(str(raw)[:500])

## 3. View Structured Extraction

The extraction stage converts raw OCR text into structured `lab_results` JSON
matching the IBM spec schema (Pydantic `LabReport` model).

Each `LabResult` contains:
- `test_name`, `test_abbreviation`, `value`, `unit`
- `reference_range` (low, high, unit)
- `flag` (HIGH / LOW / NORMAL / CRITICAL_HIGH / CRITICAL_LOW / UNKNOWN)
- `clinical_significance`

In [ ]:
if 'extraction' in result_dict:
    extraction = result_dict['extraction']
    print('=== Structured Extraction ===')
    print(json.dumps(extraction, indent=2)[:2000])
else:
    print('No extraction stage in result. Run with extraction enabled.')
    print('Available keys:', list(result_dict.keys()))

## 4. Visualize Extracted Lab Results

If lab results were extracted, let's visualize them as a table and a bar chart
of values vs. reference ranges.

In [ ]:
lab_results = []
if 'extraction' in result_dict:
    lab_results = result_dict['extraction'].get('lab_results', [])

if lab_results:
    print(f'Extracted {len(lab_results)} lab results:')
    for i, lr in enumerate(lab_results):
        print(f"  {i+1}. {lr.get('test_name', '?')}: {lr.get('value', '?')} {lr.get('unit', '')} [{lr.get('flag', '?')}]")

    # Bar chart of values
    names = [lr.get('test_name', '?') for lr in lab_results]
    values = [lr.get('value', 0) or 0 for lr in lab_results]
    flags = [lr.get('flag', 'NORMAL') for lr in lab_results]

    colors = {'NORMAL': '#4CAF50', 'HIGH': '#FF9800', 'LOW': '#2196F3',
              'CRITICAL_HIGH': '#F44336', 'CRITICAL_LOW': '#9C27B0', 'UNKNOWN': '#9E9E9E'}
    bar_colors = [colors.get(f, '#9E9E9E') for f in flags]

    plt.figure(figsize=(10, 5))
    plt.bar(names, values, color=bar_colors)
    plt.title('Extracted Lab Values')
    plt.ylabel('Value')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No lab results extracted. The LLM formatter may need an API key.')
    print('Set LLM_API_KEY in your .env file to enable structured extraction.')

## 5. Compute Evaluation Metrics (CER / WER)

The `evaluation/metrics.py` module provides:
- **CER** (Character Error Rate) — edit distance / reference length
- **WER** (Word Error Rate) — word-level edit distance
- **Exact Match Accuracy** — percentage of perfect matches

Let's compute these on a sample reference vs. hypothesis pair.

In [ ]:
# Example: compare ground truth text with OCR output
reference = 'ALT 45 U/L AST 32 U/L BILIRUBIN 0.8 mg/dL'
hypothesis = 'ALT 45 U/L AST 32 U/L BILIRUBIN 0.8 mg/dL'  # perfect match

cer = character_error_rate(reference, hypothesis)
wer = word_error_rate(reference, hypothesis)
acc = exact_match_accuracy([reference], [hypothesis])

print('=== Perfect Match Example ===')
print(f'Reference:   {reference}')
print(f'Hypothesis:   {hypothesis}')
print(f'CER:          {cer:.2f}%')
print(f'WER:          {wer:.2f}%')
print(f'Accuracy:     {acc:.2f}%')

# Now with an error
hypothesis_with_error = 'ALT 45 U/L AST 32 U/L BILIRUBIN 0.8 mg/dL'
hypothesis_with_error = 'ALT 45 U/L AST 32 U/L BILIRUBIN 0.9 mg/dL'  # typo: 0.9 vs 0.8

cer2 = character_error_rate(reference, hypothesis_with_error)
wer2 = word_error_rate(reference, hypothesis_with_error)

print(f'\n=== With Error Example ===')
print(f'Reference:   {reference}')
print(f'Hypothesis:   {hypothesis_with_error}')
print(f'CER:          {cer2:.2f}%')
print(f'WER:          {wer2:.2f}%')

## 6. Run the Benchmark Script

The `evaluation/benchmark.py` script runs the full pipeline on all sample
images in `tests/sample_images/` and generates `eval_reports/metrics_latest.json`.

You can run it from the command line:
```bash
python -m evaluation.benchmark
```

Or invoke it from this notebook:

In [ ]:
# Run the benchmark (this may take a while depending on GPU availability)
# Uncomment the lines below to execute.

# from evaluation.benchmark import main as benchmark_main
# benchmark_main()

# # Load and display the results
# metrics_path = ROOT / 'eval_reports' / 'metrics_latest.json'
# if metrics_path.exists():
#     metrics = json.loads(metrics_path.read_text())
#     print(json.dumps(metrics, indent=2))

print('Benchmark cell -- uncomment above to run the full benchmark.')
print('Results will be saved to eval_reports/metrics_latest.json')

## Summary

In this notebook we:
1. Ran the full pipeline on a sample image.
2. Inspected the raw OCR output and engine metadata.
3. Viewed the structured extraction (lab_results JSON).
4. Visualized extracted lab values with color-coded flags.
5. Computed CER and WER evaluation metrics.
6. Reviewed the benchmark script entry point.

The evaluation metrics (CER, WER) use the `jiwer` library and are
implemented in `evaluation/metrics.py`. The benchmark script generates
`eval_reports/metrics_latest.json` with aggregate metrics across all
sample images.